# Imports

In [3]:
import sys
print(sys.executable)

/home/sarah/Desktop/University/ComputerVision/SensiFake/.venv/bin/python


In [4]:
from pathlib import Path
from tqdm.auto import tqdm
import os
import random
import hashlib
import json
from uuid import uuid4
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import torch
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from huggingface_hub import hf_hub_download
import zipfile
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


# Globals

In [5]:
#=======================
# Data Configuration
#=======================
CSV_PATH = None # used for testing purposes, now the data is loaded from the kaggle dataset.
IMAGE_ROOT = None # used for testing purposes, now the data is loaded from the kaggle dataset.
OUTPUT_DIR = "checkpoints" # Output directory for saving checkpoints of the trained models. 


#=======================
# Model Configuration
#=======================
MODEL_NAME = "resnet50"
USE_PRETRAINED = True
IMAGE_SIZE = 224
RANDOM_SEED = 20
BATCH_SIZE = 64
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # DO NOT CHANGE THE WORKERS VALUE. It is set to 0 to avoid issues with Windows and Jupyter notebooks.
FREEZE_BACKBONE_EPOCHS = 1  # 0 disables initial freezing.
EARLY_STOPPING_PATIENCE = 4
TRAINING_SIZE = 0.7
VALIDATION_FRACTION = 0.1
TEST_FRACTION = 0.2
TRAINING_RUN_ID = uuid4().hex  # Reject checkpoints from previous notebook runs.

DETERMINISTIC = False  # Enable strict deterministic kernels when needed.
SENSITIVITY_NAMES = {}  # Optional, e.g. {0: "Low", 1: "High"}.


#=======================
# Device Configuration
#=======================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = not DETERMINISTIC

# optimization settigs if the system detect a GPU
PIN_MEMORY = DEVICE.type == "cuda" 
USE_AMP = DEVICE.type == "cuda"

Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


# Utils

In [6]:
# Figures and numeric results; OUTPUT_DIR retains the existing checkpoint path.
RESULTS_DIR = Path("SensiFakeProject-output")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=RANDOM_SEED, deterministic=DETERMINISTIC):
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

            
def calculate_metrics(y_true, y_pred, average="weighted"):
    """Return the common classification metrics with safe zero division."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }


# Result files and related plotting helpers.
def _save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _save_training_history(history, task):
    # The per-epoch confusion matrices are plotted separately; CSV stores scalar metrics.
    frame = pd.DataFrame({key: values for key, values in history.items()
                          if key != "confusion_matrix"})
    frame.insert(0, "epoch", np.arange(1, len(frame) + 1))
    frame.to_csv(RESULTS_DIR / f"{task.lower()}_training_history.csv", index=False)


def _save_metrics(metrics, filename):
    # Preserve metric values; undefined metrics become standard JSON null.
    def json_values(values):
        return {
            key.lower().replace(" ", "_").replace("-", "_"):
                json_values(value) if isinstance(value, dict)
                else float(value) if pd.notna(value) else None
            for key, value in values.items()
        }
    (RESULTS_DIR / filename).write_text(
        json.dumps(json_values(metrics), indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )


def plot_training_history(history, title):
    """Exactly three panels: loss, accuracy and validation F1."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set(title=f"{title} loss", xlabel="Epoch", ylabel="Loss")
    axes[0].legend()
    axes[1].plot(epochs, history["train_accuracy"], label="Train")
    axes[1].plot(epochs, history["val_accuracy"], label="Validation")
    axes[1].set(title=f"{title} accuracy", xlabel="Epoch", ylabel="Accuracy")
    axes[1].legend()
    axes[2].plot(epochs, history["val_f1"], label="Validation")
    axes[2].set(title=f"{title} F1 score", xlabel="Epoch", ylabel="F1 score")
    axes[2].legend()
    _save_figure(fig, f"{title.lower()}_training.jpg")


def plot_training_confusion(history, title, class_names):
    best_epoch = int(np.argmax(history["val_f1"]))
    matrix = history["confusion_matrix"][best_epoch]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set(title=f"{title} Best Validation Confusion Matrix (Epoch {best_epoch + 1})",
           xlabel="Predicted", ylabel="True")
    _save_figure(fig, f"{title.lower()}_training_confusion_matrix.jpg")


def save_checkpoint(path, model, task, class_names, epoch, best_metric, class_to_index):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "task": task,
        "model_name": MODEL_NAME,
        "num_classes": len(class_names),
        "class_names": list(class_names),
        "class_to_index": {str(value): int(index) for value, index in class_to_index.items()},
        "split_fingerprint": split_fingerprint,
        "training_run_id": TRAINING_RUN_ID,
        "epoch": int(epoch),
        "best_val_f1": float(best_metric),
        "model_state_dict": model.state_dict(),
    }, path)


def load_checkpoint(path, model, map_location=DEVICE):
    """Load weights while supporting PyTorch versions with/without weights_only."""
    try:
        checkpoint = torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        checkpoint = torch.load(path, map_location=map_location)
    if (checkpoint.get("split_fingerprint") != split_fingerprint
            or checkpoint.get("training_run_id") != TRAINING_RUN_ID):
        raise ValueError("Checkpoint not produced by training on the current global split/run. "
                         "Run the ADN and CSN training cells before evaluation.")
    state = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state)
    return checkpoint

def cleanup_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Data

### The following code is the download and extraction from huggingface dataset.


In [7]:
zip_path = hf_hub_download(
    repo_id="K4ru4k4i/SensiFake",
    filename="final_dataset.zip",
    repo_type="dataset"
)

extract_dir = "./sensifake"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset pronto in:", extract_dir)

path = Path(extract_dir)/"final_dataset"

Dataset pronto in: ./sensifake


### Global split 70/10/20
Un solo split condiviso da ADN e CSN. La stratificazione congiunta viene tentata su entrambe le divisioni; se non è fattibile si usa real/fake, poi uno split casuale con lo stesso seed. Il test resta escluso da training e selezione dei checkpoint.

In [8]:
metadata_files = list(path.rglob("sensifake_all.csv"))


if not metadata_files:
    raise FileNotFoundError(f"No metadata found in {path}")

metadata_path = metadata_files[0]
df = pd.read_csv(metadata_path)

Dataset_root = metadata_path.parent.parent
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Metadata: {metadata_path}")
print(f"Dataset shape: {df.shape}")
print("Columns:", list(df.columns))

# Detect label columns
label_candidates = ["label","real_fake_label"]
sensitivity_candidates = ["sensitivity","final_sensitivity_score"]

label_col = next((col for col in label_candidates if col in df.columns), None)
sensitivity_col = next((col for col in sensitivity_candidates if col in df.columns), None)

if label_col is None:
    raise ValueError(
        f"Could not identify the real/fake label column. Available columns: {list(df.columns)}"
    )

if sensitivity_col is None:
    raise ValueError("Could not identify the sensitivity label column.")
if df[[label_col, sensitivity_col]].isna().any().any():
    raise ValueError("Missing target labels: fix the metadata before splitting.")

# Preserve the original path handling before creating the shared subsets.
image_candidates = ["image", "image_path", "filepath", "file_path", "filename", "file"]
image_col = next((col for col in image_candidates if col in df.columns), None)
if image_col is None:
    raise ValueError("Could not identify the image path column.")
df["image_path"] = df[image_col].apply(lambda path: str(path))
if df["image_path"].duplicated().any():
    raise ValueError("Repeated image paths would allow image leakage between subsets.")

# Global split helpers: two divisions define a single shared 70/10/20 partition.
def _global_split(frame):
    assert np.isclose(TRAINING_SIZE + VALIDATION_FRACTION + TEST_FRACTION, 1.0)
    joint = frame[[label_col, sensitivity_col]].astype(str).agg("_".join, axis=1)
    candidates = [("real/fake + sensitivity", joint),
                  ("real/fake", frame[label_col].astype(str)), ("random", None)]
    failures = []
    for strategy, stratify_values in candidates:
        try:
            train_val, test_part = train_test_split(
                frame, test_size=TEST_FRACTION, random_state=RANDOM_SEED,
                shuffle=True, stratify=stratify_values,
            )
            remaining_strata = (None if stratify_values is None
                                else stratify_values.loc[train_val.index])
            train_part, val_part = train_test_split(
                train_val, test_size=VALIDATION_FRACTION / (1.0 - TEST_FRACTION),
                random_state=RANDOM_SEED, shuffle=True, stratify=remaining_strata,
            )
        except ValueError as exc:
            failures.append(f"{strategy}: {exc}")
            continue
        print(f"Split stratification: {strategy}")
        if failures:
            print("Fallback:", " | ".join(failures))
        return train_part, val_part, test_part
    raise ValueError("Cannot create the global split: " + " | ".join(failures))

train_df, val_df, test_df = _global_split(df)
split_indices = [set(frame.index) for frame in (train_df, val_df, test_df)]
assert not (split_indices[0] & split_indices[1] or split_indices[0] & split_indices[2]
            or split_indices[1] & split_indices[2])
assert set.union(*split_indices) == set(df.index)

# Fingerprint records subset membership only; it is never used to tune a model.
split_hasher = hashlib.sha256()
split_hasher.update(str((RANDOM_SEED, TRAINING_SIZE, VALIDATION_FRACTION, TEST_FRACTION)).encode())
for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    split_hasher.update(name.encode())
    split_hasher.update(pd.util.hash_pandas_object(
        frame[["image_path", label_col, sensitivity_col]], index=True
    ).values.tobytes())
split_fingerprint = split_hasher.hexdigest()
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Total images: {len(df)}")
split_summary = pd.DataFrame([
    {"split": name, "size": len(frame), "fraction": len(frame) / len(df),
     "real/fake": frame[label_col].value_counts().sort_index().to_dict(),
     "sensitivity": frame[sensitivity_col].value_counts().sort_index().to_dict()}
    for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df))
]).set_index("split")
print(split_summary.to_string())

# Class mappings keep the original sorted-label encoding.
adn_class_values = sorted(train_df[label_col].unique())
csn_class_values = sorted(train_df[sensitivity_col].unique())
if (set(adn_class_values) != set(df[label_col].unique())
        or set(csn_class_values) != set(df[sensitivity_col].unique())):
    raise ValueError("A class is absent from training. Cannot keep the original class-weight strategy.")
adn_class_to_index = {value: index for index, value in enumerate(adn_class_values)}
csn_class_to_index = {value: index for index, value in enumerate(csn_class_values)}
adn_class_names = [str(value) for value in adn_class_values]
csn_class_names = [SENSITIVITY_NAMES.get(value, str(value)) for value in csn_class_values]

# Release schema: authenticity 0=Real, 1=Fake; sensitivity 0=Low, 1=Medium, 2=High.
# Named labels are supported without changing their original index order.
def _semantic_indices(class_to_index, numeric_names, name_col=None):
    indices = {}
    value_names = {}
    for value, index in class_to_index.items():
        text = str(value).strip().lower()
        if text in numeric_names.values():
            name = text
        elif value in numeric_names:
            name = numeric_names[value]
        elif text in {str(key): name for key, name in numeric_names.items()}:
            name = {str(key): name for key, name in numeric_names.items()}[text]
        else:
            raise ValueError(f"Unknown label meaning: {value!r}; check the dataset schema.")
        if name in indices:
            raise ValueError(f"Multiple label values represent {name}.")
        indices[name] = index
        value_names[value] = name
    assert set(indices) == set(numeric_names.values()), "Missing or unexpected classes."
    if name_col is not None and name_col in df.columns:
        target_col = label_col if name_col == "real_fake_label_name" else sensitivity_col
        expected = df[target_col].map(value_names)
        assert expected.equals(df[name_col].astype(str).str.strip().str.lower()), (
            f"Numeric labels disagree with {name_col}; do not reinterpret their meaning."
        )
    return indices

adn_indices = _semantic_indices(adn_class_to_index, {0: "real", 1: "fake"}, "real_fake_label_name")
csn_indices = _semantic_indices(csn_class_to_index, {0: "low", 1: "medium", 2: "high"}, "final_sensitivity_label")
adn_display_names = [next(name.title() for name, index in adn_indices.items() if index == i)
                     for i in range(len(adn_class_to_index))]
csn_display_names = [next(name.title() for name, index in csn_indices.items() if index == i)
                     for i in range(len(csn_class_to_index))]
print("ADN class_to_index:", adn_class_to_index, "semantic indices:", adn_indices)
print("CSN class_to_index:", csn_class_to_index, "semantic indices:", csn_indices)

Metadata: sensifake/final_dataset/metadata/sensifake_all.csv
Dataset shape: (4499, 29)
Columns: ['image_id', 'image_path', 'source_dataset', 'source_original_path', 'source_original_paths_json', 'sha256', 'source_dataset_url', 'source_dataset_license', 'source_original_id', 'source_requested_revision', 'source_resolved_revision', 'real_fake_label', 'real_fake_label_name', 'real_fake_label_provenance', 'annotation_type', 'sensitivity_label_quality', 'human_annotation_available', 'human_annotation_source', 'human_sensitivity_score', 'human_sensitivity_label', 'predicted_sensitivity_score', 'predicted_sensitivity_label', 'p_ge_1', 'p_ge_2', 'final_sensitivity_score', 'final_sensitivity_label', 'model_version', 'checkpoint_sha256', 'v3_supervision_split']
Split stratification: real/fake + sensitivity
Total images: 4499
            size  fraction           real/fake                sensitivity
split                                                                    
train       3149  0.69993

# Network

### The following code define the model architecture and the training loop. It also includes functions for evaluating the model on the validation set and saving the best model based on validation performance.

In [9]:
def create_base_model(num_classes, pretrained=USE_PRETRAINED):
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    try:
        network = models.resnet50(weights=weights)
    except Exception as exc:
        if pretrained:
            raise RuntimeError(
                "Failed to load pretrained weights. Set USE_PRETRAINED=False to initialize randomly."
            ) from exc
        raise
    network.fc = nn.Linear(network.fc.in_features, num_classes)
    return network


class ADN(nn.Module):
    """Independent image-only real/fake classifier with two output logits."""
    def __init__(self, pretrained=USE_PRETRAINED):
        super().__init__()
        self.backbone = create_base_model(num_classes=2, pretrained=pretrained)

    def forward(self, image):
        return self.backbone(image)

class CSN(nn.Module):
    """Independent image-only sensitivity classifier."""
    def __init__(self, num_sensitivity_classes, pretrained=USE_PRETRAINED):
        super().__init__()
        if num_sensitivity_classes < 2:
            raise ValueError("At least two sensitivity classes are required for classification.")
        self.backbone = create_base_model(num_classes=num_sensitivity_classes, pretrained=pretrained)

    def forward(self, image):
        return self.backbone(image)

    

# Train

#### Training function and class definition

In [10]:
class ImageClassificationDataset(Dataset):
    def __init__(self, frame, target_col, class_to_index, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.target_col = target_col
        self.class_to_index = class_to_index
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = Dataset_root / str(row["image_path"])

        try:
            image = Image.open(image_path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError) as exc:
            raise RuntimeError(f"Could not load image: {image_path}") from exc

        if self.transform:
            image = self.transform(image)

        label = self.class_to_index[row[self.target_col]]
        return image, torch.tensor(label, dtype=torch.long)


def _make_transforms():
    weights = models.ResNet50_Weights.DEFAULT
    preprocess = weights.transforms()

    normalize = transforms.Normalize(
        mean=preprocess.mean,
        std=preprocess.std,
    )

    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        normalize,
    ])

    return train_transform, eval_transform


# DataLoader helpers: receive the global subsets without splitting again.
def _build_loaders(train_part, val_part, target_col, class_to_index):
    train_transform, eval_transform = _make_transforms()
    train_dataset = ImageClassificationDataset(
        train_part, target_col, class_to_index, train_transform
    )
    val_dataset = ImageClassificationDataset(
        val_part, target_col, class_to_index, eval_transform
    )
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    return train_loader, val_loader


def _build_test_loader(frame, target_col, class_to_index):
    _, eval_transform = _make_transforms()
    dataset = ImageClassificationDataset(frame, target_col, class_to_index, eval_transform)
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )


def _class_weights(frame, target_col, class_to_index):
    labels = frame[target_col].map(class_to_index).to_numpy()
    classes = np.arange(len(class_to_index))

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=labels,
    )
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def _set_backbone_trainable(model, trainable):
    for parameter in model.backbone.parameters():
        parameter.requires_grad = trainable

    for parameter in model.backbone.fc.parameters():
        parameter.requires_grad = True


def _evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())

    metrics = calculate_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def _train_classifier(model, train_loader, val_loader, criterion, task, class_names, class_to_index):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scaler = GradScaler(enabled=USE_AMP)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
        "val_f1": [],
        "confusion_matrix": [],
    }

    best_f1 = -np.inf
    best_epoch = 0
    patience_counter = 0
    checkpoint_path = Path(OUTPUT_DIR) / f"{task.lower()}_best.pt"

    checkpoint_saved = False
    for epoch in range(NUM_EPOCHS):
        if epoch == FREEZE_BACKBONE_EPOCHS:
            _set_backbone_trainable(model, True)
            optimizer = torch.optim.AdamW(
                filter(lambda parameter: parameter.requires_grad, model.parameters()),
                lr=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY,
            )

        model.train()
        running_loss = 0.0
        y_true, y_pred = [], []

        progress = tqdm(train_loader, desc=f"{task} epoch {epoch + 1}/{NUM_EPOCHS}")

        for images, labels in progress:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * labels.size(0)
            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(logits.argmax(dim=1).detach().cpu().numpy())

        train_metrics = calculate_metrics(y_true, y_pred)
        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = _evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_f1"].append(val_metrics["f1"])
        history["confusion_matrix"].append(val_metrics["confusion_matrix"])
        print(
            f"Train loss: {train_loss:.4f}, "
            f"Train accuracy: {train_metrics['accuracy']:.4f}, "
            f"Val loss: {val_metrics['loss']:.4f}, "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_epoch = epoch + 1
            patience_counter = 0
            save_checkpoint(
                checkpoint_path,
                model,
                task,
                class_names,
                best_epoch,
                best_f1,
                class_to_index,
            )
            checkpoint_saved = True
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print("Early stopping.")
                break

    if not checkpoint_saved:
        raise RuntimeError(f"{task}: no valid best checkpoint was saved in this training run.")
    load_checkpoint(checkpoint_path, model)
    return model, history, checkpoint_path


def train_adn():
    class_to_index = adn_class_to_index
    class_names = adn_class_names
    train_loader, val_loader = _build_loaders(
        train_df, val_df, label_col, class_to_index
    )

    model = ADN(pretrained=USE_PRETRAINED)
    _set_backbone_trainable(model, FREEZE_BACKBONE_EPOCHS == 0)

    criterion = nn.CrossEntropyLoss(
        weight=_class_weights(train_df, label_col, class_to_index)
    )

    return _train_classifier(
        model, train_loader, val_loader, criterion, "ADN", class_names, class_to_index
    )


def train_csn():
    class_values = csn_class_values
    class_to_index = csn_class_to_index
    class_names = csn_class_names
    train_loader, val_loader = _build_loaders(
        train_df, val_df, sensitivity_col, class_to_index
    )

    model = CSN(
        num_sensitivity_classes=len(class_values),
        pretrained=USE_PRETRAINED,
    )
    _set_backbone_trainable(model, FREEZE_BACKBONE_EPOCHS == 0)

    criterion = nn.CrossEntropyLoss(
        weight=_class_weights(train_df, sensitivity_col, class_to_index)
    )

    return _train_classifier(
        model, train_loader, val_loader, criterion, "CSN", class_names, class_to_index
    )

# Evaluation

In [11]:
# Train both classifiers
set_seed()

adn_model, adn_history, adn_checkpoint = train_adn()

cleanup_gpu_memory()

csn_model, csn_history, csn_checkpoint = train_csn()


print(f"ADN checkpoint: {adn_checkpoint}")
print(f"CSN checkpoint: {csn_checkpoint}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/sarah/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:26<00:00, 3.85MB/s]
/tmp/ipykernel_505330/806679101.py:131: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)
ADN epoch 1/10:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_505330/806679101.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
ADN epoch 1/10: 100%|██████████| 50/50 [01:25<00:00,  1.71s/it]
/tmp/ipykernel_505330/806679101.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Train loss: 0.6794, Train accuracy: 0.5627, Val loss: 0.6563, Val F1: 0.6755


ADN epoch 2/10:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_505330/806679101.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
ADN epoch 2/10: 100%|██████████| 50/50 [01:28<00:00,  1.78s/it]
/tmp/ipykernel_505330/806679101.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Train loss: 0.4678, Train accuracy: 0.7939, Val loss: 0.3743, Val F1: 0.8261


...

Train loss: 0.0157, Train accuracy: 0.9962, Val loss: 1.1522, Val F1: 0.8107


CSN epoch 9/10:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_505330/806679101.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
CSN epoch 9/10: 100%|██████████| 50/50 [01:19<00:00,  1.60s/it]
/tmp/ipykernel_505330/806679101.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Train loss: 0.0094, Train accuracy: 0.9971, Val loss: 1.1533, Val F1: 0.8267


CSN epoch 10/10:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_505330/806679101.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
CSN epoch 10/10: 100%|██████████| 50/50 [01:21<00:00,  1.63s/it]
/tmp/ipykernel_505330/806679101.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Train loss: 0.0067, Train accuracy: 0.9987, Val loss: 1.2928, Val F1: 0.8222
ADN checkpoint: checkpoints/adn_best.pt
CSN checkpoint: checkpoints/csn_best.pt


#### Little script to see witch parameters are saved in the training loop

In [12]:
for name, history in [("ADN", adn_history), ("CSN", csn_history)]:
    print(f"\n{name}")
    _save_training_history(history, name)
    for key, values in history.items():
        print(f"  {key}: {len(values)} values")



ADN
  train_loss: 8 values
  val_loss: 8 values
  train_accuracy: 8 values
  val_accuracy: 8 values
  val_f1: 8 values
  confusion_matrix: 8 values

CSN
  train_loss: 10 values
  val_loss: 10 values
  train_accuracy: 10 values
  val_accuracy: 10 values
  val_f1: 10 values
  confusion_matrix: 10 values


### Code for plotting the models metrics

In [ ]:
plot_training_history(adn_history, "ADN")
plot_training_confusion(adn_history, "ADN", adn_display_names)
plot_training_history(csn_history, "CSN")
plot_training_confusion(csn_history, "CSN", csn_display_names)


### Best checkpoints from the current training run
I file mantengono i path originali. Ogni checkpoint deve corrispondere allo split globale, ai mapping e al training appena eseguito; i checkpoint precedenti non sono accettati.

In [14]:
adn_checkpoint = Path(OUTPUT_DIR) / "adn_best.pt"
csn_checkpoint = Path(OUTPUT_DIR) / "csn_best.pt"

if not adn_checkpoint.exists():
    raise FileNotFoundError(f"ADN checkpoint not found: {adn_checkpoint}")

if not csn_checkpoint.exists():
    raise FileNotFoundError(f"CSN checkpoint not found: {csn_checkpoint}")

# Recreate models without downloading pretrained weights, then validate checkpoint provenance.
adn_model = ADN(pretrained=False).to(DEVICE)
csn_model = CSN(num_sensitivity_classes=len(csn_class_to_index), pretrained=False).to(DEVICE)
adn_metadata = load_checkpoint(adn_checkpoint, adn_model)
csn_metadata = load_checkpoint(csn_checkpoint, csn_model)

for task, metadata, mapping, names in (
    ("ADN", adn_metadata, adn_class_to_index, adn_class_names),
    ("CSN", csn_metadata, csn_class_to_index, csn_class_names),
):
    assert metadata["task"] == task and metadata["num_classes"] == len(mapping)
    assert metadata["class_names"] == names
    assert metadata["class_to_index"] == {str(value): int(index) for value, index in mapping.items()}

adn_model.eval()
csn_model.eval()

print(f"Loaded ADN checkpoint: {adn_checkpoint}")
print(f"ADN classes: {adn_metadata['class_names']}")
print(f"Loaded CSN checkpoint: {csn_checkpoint}")
print(f"CSN classes: {csn_metadata['class_names']}")

Loaded ADN checkpoint: checkpoints/adn_best.pt
ADN classes: ['0', '1']
Loaded CSN checkpoint: checkpoints/csn_best.pt
CSN classes: ['0', '1', '2']


## Test Set Evaluation

In [ ]:
# Test inference: preserve loader order and collect every sample.
def _test_inference(model, loader):
    model.eval()
    y_true, probabilities = [], []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = model(images)
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
            y_true.append(labels.cpu().numpy())
    y_true = np.concatenate(y_true)
    probabilities = np.concatenate(probabilities)
    assert len(y_true) == len(probabilities) == len(loader.dataset)
    return y_true, probabilities.argmax(axis=1), probabilities


# Metrics: class indices always come from the explicit mappings.
def _adn_test_metrics(y_true, y_pred):
    labels = list(range(len(adn_class_to_index)))
    fake_index, real_index = adn_indices["fake"], adn_indices["real"]
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    fake_count = matrix[fake_index].sum()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "Fake Precision": precision_score(y_true, y_pred, pos_label=fake_index, zero_division=0),
        "Fake Recall": (recall_score(y_true, y_pred, pos_label=fake_index, zero_division=0)
                        if fake_count else np.nan),
        "Fake F1": f1_score(y_true, y_pred, pos_label=fake_index, zero_division=0),
        "Fake False Negative Rate": (matrix[fake_index, real_index] / fake_count
                                     if fake_count else np.nan),
    }


def _plot_test_confusion(y_true, y_pred, names, title, ax, normalize=None):
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(names))), normalize=normalize)
    options = ({"vmin": 0, "vmax": 1, "cbar_kws": {"label": "Fraction of true class"}}
               if normalize else {})
    sns.heatmap(matrix, annot=True, fmt=".2f" if normalize else "d", cmap="Blues", ax=ax,
                xticklabels=names, yticklabels=names, **options)
    ax.set(title=title, xlabel="Predicted", ylabel="True")


adn_test_loader = _build_test_loader(test_df, label_col, adn_class_to_index)
csn_test_loader = _build_test_loader(test_df, sensitivity_col, csn_class_to_index)
assert len(adn_test_loader.dataset) == len(csn_test_loader.dataset) == len(test_df)
assert adn_test_loader.dataset.frame["image_path"].equals(csn_test_loader.dataset.frame["image_path"])
adn_y_true, adn_y_pred, adn_test_probabilities = _test_inference(adn_model, adn_test_loader)
csn_y_true, csn_y_pred, csn_test_probabilities = _test_inference(csn_model, csn_test_loader)
assert np.array_equal(adn_y_true, test_df[label_col].map(adn_class_to_index).to_numpy())
assert np.array_equal(csn_y_true, test_df[sensitivity_col].map(csn_class_to_index).to_numpy())

adn_test_metrics = _adn_test_metrics(adn_y_true, adn_y_pred)
print("ADN test metrics:")
print(pd.DataFrame([adn_test_metrics], index=["ADN"]).round(4).to_string())
csn_test_metrics = {
    "Accuracy": accuracy_score(csn_y_true, csn_y_pred),
    "Macro F1": f1_score(csn_y_true, csn_y_pred, labels=list(range(len(csn_class_to_index))),
                          average="macro", zero_division=0),
    "Weighted F1": f1_score(csn_y_true, csn_y_pred, labels=list(range(len(csn_class_to_index))),
                             average="weighted", zero_division=0),
}
print("\nCSN test metrics:")
print(pd.DataFrame([csn_test_metrics], index=["CSN"]).round(4).to_string())
csn_report = classification_report(
    csn_y_true, csn_y_pred, labels=list(range(len(csn_class_to_index))),
    target_names=csn_display_names, output_dict=True, zero_division=0,
)
print(pd.DataFrame(csn_report).T.loc[csn_display_names, ["precision", "recall", "f1-score", "support"]]
      .round(4).to_string())
# Save the unrounded metrics already computed above.
_save_metrics(adn_test_metrics, "adn_test_metrics.json")
_save_metrics({**csn_test_metrics, **{name: csn_report[name] for name in csn_display_names}},
              "csn_test_metrics.json")

# Dedicated test confusion matrices; ADN is normalized by true class.
fig, ax = plt.subplots(figsize=(6, 5))
_plot_test_confusion(adn_y_true, adn_y_pred, adn_display_names, "ADN — Test", ax, normalize="true")
_save_figure(fig, "adn_test_confusion_matrix.jpg")
fig, ax = plt.subplots(figsize=(6, 5))
_plot_test_confusion(csn_y_true, csn_y_pred, csn_display_names, "CSN — Test", ax)
_save_figure(fig, "csn_test_confusion_matrix.jpg")

# ROC uses the original Fake probabilities, without modifying predictions.
adn_fake_true = (adn_y_true == adn_indices["fake"]).astype(int)
if np.unique(adn_fake_true).size == 2:
    false_positive_rate, true_positive_rate, _ = roc_curve(
        adn_fake_true, adn_test_probabilities[:, adn_indices["fake"]]
    )
    adn_roc_auc = roc_auc_score(adn_fake_true, adn_test_probabilities[:, adn_indices["fake"]])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(false_positive_rate, true_positive_rate, label=f"ADN (AUC = {adn_roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
    ax.set(title="ADN test ROC — fake positive class", xlabel="False positive rate",
           ylabel="True positive rate (fake recall)", xlim=(0, 1), ylim=(0, 1))
    ax.legend()
    _save_figure(fig, "adn_roc.jpg")
else:
    print("ROC curve unavailable: the test set contains only one authenticity class.")

# ADN performance grouped by true sensitivity, reusing the same test predictions.
adn_by_sensitivity = []
for name in ("low", "medium", "high"):
    mask = csn_y_true == csn_indices[name]
    count = int(mask.sum())
    metrics = _adn_test_metrics(adn_y_true[mask], adn_y_pred[mask]) if count else {}
    adn_by_sensitivity.append({
        "Sensitivity": name.title(), "Samples": count,
        **{metric: metrics.get(metric, np.nan) for metric in
           ("Accuracy", "Macro F1", "Fake Recall", "Fake False Negative Rate")},
    })
adn_by_sensitivity = pd.DataFrame(adn_by_sensitivity).set_index("Sensitivity")
print("\nADN by true sensitivity:")
print(adn_by_sensitivity.round(4).to_string())
ax = adn_by_sensitivity[["Accuracy", "Macro F1", "Fake Recall"]].plot.bar(
    figsize=(8, 5), rot=0, ylim=(0, 1)
)
ax.set(title="ADN test performance by sensitivity", xlabel="True sensitivity", ylabel="Score")
ax.legend(title="Metric")
_save_figure(ax.figure, "adn_by_sensitivity.jpg")
ax = adn_by_sensitivity["Fake False Negative Rate"].plot.bar(figsize=(6, 4), rot=0, ylim=(0, 1))
ax.set(title="ADN fake false-negative rate by sensitivity", xlabel="True sensitivity",
       ylabel="Fraction of fake images predicted as real")
_save_figure(ax.figure, "adn_fnr_by_sensitivity.jpg")


## Fusion / Final Evaluation
$$s = 0.93\,P(High) + 0.50\,P(Medium) + 0.30\,P(Low),\qquad
\tau = 0.5 + 0.40\,s.$$
**Real** se $P(Real) > \tau$; **Fake** altrimenti, anche in caso di uguaglianza.


In [ ]:
W_H = 0.93
W_M = 0.50
W_L = 0.30
EPSILON = 0.40

# Class meanings are validated in Data and when loading the checkpoints.
assert set(adn_indices) == {"real", "fake"}
assert set(csn_indices) == {"low", "medium", "high"}


def apply_fusion(adn_probabilities, csn_probabilities):
    p_real = adn_probabilities[..., adn_indices["real"]]
    p_fake = adn_probabilities[..., adn_indices["fake"]]
    low_value = csn_probabilities[..., csn_indices["low"]]
    medium_value = csn_probabilities[..., csn_indices["medium"]]
    high_value = csn_probabilities[..., csn_indices["high"]]
    prudence_score = (
        high_value * W_H + medium_value * W_M + low_value * W_L
    )
    threshold = 0.5 + prudence_score * EPSILON
    result = np.where(p_real > threshold, "real", "fake")
    return prudence_score, threshold, result


prudence_score, threshold, res = apply_fusion(adn_test_probabilities, csn_test_probabilities)
fusion_y_pred = np.where(res == "real", adn_indices["real"], adn_indices["fake"])
identifier_columns = [col for col in ("image_id", "image_path") if col in test_df.columns]
test_results_df = test_df[identifier_columns].copy()
test_results_df["true_real_fake"] = test_df[label_col].to_numpy()
test_results_df["true_sensitivity"] = test_df[sensitivity_col].to_numpy()
test_results_df["p_real"] = adn_test_probabilities[:, adn_indices["real"]]
test_results_df["p_fake"] = adn_test_probabilities[:, adn_indices["fake"]]
test_results_df["p_low"] = csn_test_probabilities[:, csn_indices["low"]]
test_results_df["p_medium"] = csn_test_probabilities[:, csn_indices["medium"]]
test_results_df["p_high"] = csn_test_probabilities[:, csn_indices["high"]]
test_results_df["prudence_score"] = prudence_score
test_results_df["fusion_threshold"] = threshold
adn_index_to_value = {index: value for value, index in adn_class_to_index.items()}
test_results_df["adn_prediction"] = [adn_index_to_value[int(index)] for index in adn_y_pred]
test_results_df["fusion_prediction"] = [adn_index_to_value[int(index)] for index in fusion_y_pred]
assert len(test_results_df) == len(test_df) == len(adn_y_true) == len(fusion_y_pred)
assert test_results_df["image_path"].equals(test_df["image_path"])
test_results_df.to_csv(RESULTS_DIR / "fusion_predictions.csv", index=False)
print(f"Fusion evaluated on all {len(test_results_df)} test images.")
print(test_results_df.head().round(4).to_string(index=False))


## ADN vs Sensitivity-Aware Fusion
Confronto sullo stesso test set, con parametri di fusion fissi. Le metriche per sensitivity usano la vera label; Recall/FNR sono `NaN` se mancano campioni Fake.

In [ ]:
# Both predictions refer to the same ordered test_df; no recalibration follows.
fusion_test_metrics = _adn_test_metrics(adn_y_true, fusion_y_pred)
comparison_columns = ["Accuracy", "Macro F1", "Fake Precision", "Fake Recall", "Fake F1",
                      "Fake False Negative Rate"]
adn_vs_fusion = pd.DataFrame(
    [adn_test_metrics, fusion_test_metrics],
    index=["ADN alone", "Fusion"],
)[comparison_columns]
print(adn_vs_fusion.round(4).to_string())
_save_metrics(fusion_test_metrics, "fusion_test_metrics.json")
adn_vs_fusion.to_csv(RESULTS_DIR / "adn_vs_fusion.csv", index_label="Detector")

# ADN's confusion matrix is already displayed in Test Set Evaluation.
fig, ax = plt.subplots(figsize=(6, 5))
_plot_test_confusion(adn_y_true, fusion_y_pred, adn_display_names, "Fusion — Test", ax)
_save_figure(fig, "fusion_test_confusion_matrix.jpg")

# Group by true sensitivity, not by CSN prediction.
fusion_by_sensitivity = []
for name in ("low", "medium", "high"):
    mask = csn_y_true == csn_indices[name]
    count = int(mask.sum())
    metrics = _adn_test_metrics(adn_y_true[mask], fusion_y_pred[mask]) if count else {}
    fusion_by_sensitivity.append({
        "Sensitivity": name.title(), "Samples": count,
        "Accuracy": metrics.get("Accuracy", np.nan),
        "Fake Recall": metrics.get("Fake Recall", np.nan),
        "Fake False Negative Rate": metrics.get("Fake False Negative Rate", np.nan),
    })
fusion_by_sensitivity = pd.DataFrame(fusion_by_sensitivity).set_index("Sensitivity")
assert fusion_by_sensitivity["Samples"].sum() == len(test_df)
print("\nFusion by true sensitivity:")
print(fusion_by_sensitivity.round(4).to_string())
print("NaN: empty group or no true Fake samples for Recall/FNR.")
fusion_by_sensitivity.to_csv(RESULTS_DIR / "fusion_by_sensitivity.csv", index_label="Sensitivity")

# One compact comparison highlights the change in Fake recall and missed Fakes.
ax = adn_vs_fusion[["Fake Recall", "Fake False Negative Rate"]].T.plot.bar(
    figsize=(7, 4), rot=0, ylim=(0, 1)
)
ax.set(title="ADN vs Sensitivity-Aware Fusion", xlabel="Metric", ylabel="Score / Rate")
ax.legend(title="Detector")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", label_type="center")
_save_figure(ax.figure, "adn_vs_fusion.jpg")


### Single-image demo
Usa `test_image.jpg` nella root del progetto e gli stessi modelli e la stessa fusion della valutazione sul test.


In [ ]:
DEMO_IMAGE_PATH = Path("test_image.jpg")
if not DEMO_IMAGE_PATH.is_file():
    print(f"Demo skipped: {DEMO_IMAGE_PATH} not found in the project root.")
else:
    _, eval_transform = _make_transforms()
    demo_image = Image.open(DEMO_IMAGE_PATH).convert("RGB")
    demo_tensor = eval_transform(demo_image).unsqueeze(0).to(DEVICE)

    # The best models are already in evaluation mode; no checkpoint reload or model update.
    with torch.inference_mode():
        with autocast(enabled=USE_AMP):
            demo_adn_logits = adn_model(demo_tensor)
            demo_csn_logits = csn_model(demo_tensor)
    demo_adn_probabilities = torch.softmax(demo_adn_logits, dim=1)[0].cpu().numpy()
    demo_csn_probabilities = torch.softmax(demo_csn_logits, dim=1)[0].cpu().numpy()
    demo_adn_index = int(demo_adn_probabilities.argmax())
    demo_csn_index = int(demo_csn_probabilities.argmax())

    print(f"Image: {DEMO_IMAGE_PATH}")
    print(f"ADN prediction: {adn_display_names[demo_adn_index]}")
    print("ADN probabilities:")
    for name in ("real", "fake"):
        print(f"  {name.title()}: {demo_adn_probabilities[adn_indices[name]]:.2%}")
    print(f"\nCSN prediction: {csn_display_names[demo_csn_index]}")
    print("CSN probabilities:")
    for name in ("low", "medium", "high"):
        print(f"  {name.title()}: {demo_csn_probabilities[csn_indices[name]]:.2%}")

    demo_prudence_score, demo_threshold, demo_result = apply_fusion(
        demo_adn_probabilities, demo_csn_probabilities
    )
    print(f"\nPrudence score: {demo_prudence_score}")
    print(f"Fusion threshold: {demo_threshold}")
    print(f"Final fusion result: {demo_result.item()}")
